# MS1MV3

## 데이터 수집

In [21]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("trnguyncng/ms1mv3")

print("Path to dataset files:", path)

100%|██████████| 24.2G/24.2G [04:03<00:00, 107MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/trnguyncng/ms1mv3/versions/1


In [22]:
! ls -al /root/.cache/kagglehub/datasets/trnguyncng/ms1mv3/versions/1/ms1m-retinaface-t1

total 29155716
drwxr-xr-x 2 root root        4096 May  9 18:05 .
drwxr-xr-x 3 root root        4096 May  9 18:05 ..
-rw-r--r-- 1 root root   102395035 May  9 18:05 train.idx
-rw-r--r-- 1 root root   430974328 May  9 18:05 train.lst
-rw-r--r-- 1 root root 29322062124 May  9 18:11 train.rec


In [23]:
"""
MS1MV3 데이터셋 다운로드 검증 스크립트
경로: /root/.cache/kagglehub/datasets/trnguyncng/ms1mv3/versions/1
"""

import os
import struct
from pathlib import Path

DATASET_PATH = Path("/root/.cache/kagglehub/datasets/trnguyncng/ms1mv3/versions/1")


def check_directory_structure():
    """전체 디렉토리 구조 출력"""
    print("=" * 60)
    print("1. 디렉토리 구조 확인")
    print("=" * 60)

    if not DATASET_PATH.exists():
        print(f"❌ 경로가 존재하지 않습니다: {DATASET_PATH}")
        return False

    print(f"✅ 경로 확인: {DATASET_PATH}\n")

    # 최상위 파일/폴더 목록
    items = sorted(DATASET_PATH.iterdir())
    print(f"최상위 항목 ({len(items)}개):")
    for item in items:
        size = get_size(item)
        tag = "📁" if item.is_dir() else "📄"
        print(f"  {tag} {item.name:<40} {size}")

    return True


def get_size(path: Path) -> str:
    """파일/폴더 크기 반환"""
    if path.is_file():
        size = path.stat().st_size
    else:
        size = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())

    for unit in ["B", "KB", "MB", "GB"]:
        if size < 1024:
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} TB"


def check_rec_files():
    """RecordIO 파일 존재 및 기본 검증"""
    print("\n" + "=" * 60)
    print("2. RecordIO 파일 검증 (.rec / .idx)")
    print("=" * 60)

    rec_files = list(DATASET_PATH.rglob("*.rec"))
    idx_files = list(DATASET_PATH.rglob("*.idx"))

    if not rec_files:
        print("⚠️  .rec 파일 없음 → 폴더 형태 데이터셋일 수 있음")
        return False

    for rec_path in rec_files:
        idx_path = rec_path.with_suffix(".idx")
        print(f"\n  .rec : {rec_path}")
        print(f"  .idx : {idx_path}")
        print(f"  크기 : rec={get_size(rec_path)}, idx={get_size(idx_path) if idx_path.exists() else '없음'}")

        # .idx 파싱으로 레코드 수 계산
        if idx_path.exists():
            count = count_records_from_idx(idx_path)
            print(f"  레코드 수: {count:,}장")
        else:
            print("  ❌ .idx 파일 없음")

    return True


def count_records_from_idx(idx_path: Path) -> int:
    """idx 파일에서 레코드 수 계산 (MXNet 없이)"""
    count = 0
    try:
        with open(idx_path, "rb") as f:
            while True:
                buf = f.read(8)  # key(4) + offset(4) or key(8) + offset(8)
                if not buf:
                    break
                count += 1
        # 각 줄이 "key\toffset\n" 텍스트 형식인 경우도 시도
    except Exception:
        pass

    # 텍스트 형식 시도
    if count == 0:
        try:
            with open(idx_path, "r") as f:
                count = sum(1 for line in f if line.strip())
        except Exception:
            pass

    return count


def check_folder_structure():
    """폴더 형태 데이터셋 검증"""
    print("\n" + "=" * 60)
    print("3. 폴더 형태 검증 (identity_id/image.jpg)")
    print("=" * 60)

    # 숫자 이름의 폴더 찾기
    subdirs = [d for d in DATASET_PATH.iterdir() if d.is_dir()]

    if not subdirs:
        print("  폴더 없음")
        return False

    print(f"  총 아이덴티티 폴더 수: {len(subdirs):,}")

    # 샘플 5개 확인
    sample_dirs = sorted(subdirs)[:5]
    print(f"\n  샘플 5개 확인:")
    img_counts = []
    for d in sorted(subdirs):
        imgs = list(d.glob("*.jpg")) + list(d.glob("*.png")) + list(d.glob("*.jpeg"))
        img_counts.append(len(imgs))

    for d, cnt in zip(sorted(subdirs)[:5], img_counts[:5]):
        print(f"    📁 {d.name}: {cnt}장")

    if img_counts:
        print(f"\n  전체 통계:")
        print(f"    총 이미지 수   : {sum(img_counts):,}장")
        print(f"    인물당 평균    : {sum(img_counts)/len(img_counts):.1f}장")
        print(f"    인물당 최소    : {min(img_counts)}장")
        print(f"    인물당 최대    : {max(img_counts)}장")
        print(f"    100장 이상 인물: {sum(1 for c in img_counts if c >= 100):,}명")

    return True


def check_bin_files():
    """평가용 .bin 파일 확인 (LFW, CFP 등)"""
    print("\n" + "=" * 60)
    print("4. 평가 벤치마크 .bin 파일 확인")
    print("=" * 60)

    bin_files = list(DATASET_PATH.rglob("*.bin"))
    if not bin_files:
        print("  .bin 파일 없음 (학습 데이터만 포함된 버전)")
        return

    for b in bin_files:
        print(f"  ✅ {b.name:<30} {get_size(b)}")


def check_property_file():
    """property 파일 확인 (id 수, 이미지 수 메타데이터)"""
    print("\n" + "=" * 60)
    print("5. property 파일 확인")
    print("=" * 60)

    prop_files = list(DATASET_PATH.rglob("property"))
    if not prop_files:
        print("  property 파일 없음")
        return

    for p in prop_files:
        print(f"  {p}:")
        print(f"  내용: {p.read_text().strip()}")


def main():
    print(f"\n🔍 MS1MV3 데이터셋 검증 시작")
    print(f"   경로: {DATASET_PATH}\n")

    ok = check_directory_structure()
    if not ok:
        return

    has_rec = check_rec_files()
    if not has_rec:
        check_folder_structure()

    check_bin_files()
    check_property_file()

    print("\n" + "=" * 60)
    print("✅ 검증 완료")
    print("=" * 60)
    print("\n다음 단계:")
    print("  → .rec 형태: sample_ms1mv3.py에서 dataset_type='rec' 설정")
    print("  → 폴더 형태: sample_ms1mv3.py에서 dataset_type='folder' 설정")


if __name__ == "__main__":
    main()


🔍 MS1MV3 데이터셋 검증 시작
   경로: /root/.cache/kagglehub/datasets/trnguyncng/ms1mv3/versions/1

1. 디렉토리 구조 확인
✅ 경로 확인: /root/.cache/kagglehub/datasets/trnguyncng/ms1mv3/versions/1

최상위 항목 (1개):
  📁 ms1m-retinaface-t1                       27.8 GB

2. RecordIO 파일 검증 (.rec / .idx)

  .rec : /root/.cache/kagglehub/datasets/trnguyncng/ms1mv3/versions/1/ms1m-retinaface-t1/train.rec
  .idx : /root/.cache/kagglehub/datasets/trnguyncng/ms1mv3/versions/1/ms1m-retinaface-t1/train.idx
  크기 : rec=27.3 GB, idx=97.7 MB
  레코드 수: 12,799,380장

4. 평가 벤치마크 .bin 파일 확인
  .bin 파일 없음 (학습 데이터만 포함된 버전)

5. property 파일 확인
  property 파일 없음

✅ 검증 완료

다음 단계:
  → .rec 형태: sample_ms1mv3.py에서 dataset_type='rec' 설정
  → 폴더 형태: sample_ms1mv3.py에서 dataset_type='folder' 설정


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# .rec 파일 처리를 위한 라이브러리?
! pip install mxnet

In [ ]:
"""
MS1MV3 데이터셋에서 50인 랜덤 추출 스크립트
----------------------------------------------
MS1MV3는 두 가지 형태로 배포됩니다:
  1. .rec / .idx 형태 (MXNet RecordIO) ← 가장 일반적
  2. 압축 해제된 폴더 형태 (identity_id/image.jpg)

이 스크립트는 두 형태 모두 지원합니다.
사용 전 설정: CONFIG 섹션의 경로와 파라미터를 수정하세요.
"""

import os
import random
import shutil
import struct
from pathlib import Path

import numpy as np
# START: Fix for mxnet/numpy compatibility issue
# This is a workaround for mxnet versions that rely on deprecated np.bool.
# It ensures that 'np.bool' exists if it's missing in newer NumPy versions.
if not hasattr(np, 'bool'):
    np.bool = bool
# END: Fix for mxnet/numpy compatibility issue

# ─────────────────────────────────────────────
# CONFIG: 본인 환경에 맞게 수정
# ─────────────────────────────────────────────
CONFIG = {
    # 데이터셋 형태: "rec" 또는 "folder"
    "dataset_type": "rec",

    # [rec 형태] .rec, .idx 파일이 있는 디렉토리
    "rec_dir": "/root/.cache/kagglehub/datasets/trnguyncng/ms1mv3/versions/1/ms1m-retinaface-t1",          # 예: ~/datasets/ms1mv3

    # # [folder 형태] 압축 해제된 루트 디렉토리
    # # 구조: root_dir/identity_id/image.jpg
    # "folder_dir": "/path/to/ms1mv3_imgs",

    # 출력 디렉토리 (복사된 이미지 저장)
    "output_dir": "/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/ms1mv3",

    # 추출 조건
    "num_persons": 50,          # 추출할 인물 수
    "min_images": 100,          # 인물당 최소 이미지 수 (품질 필터)
    "images_per_person": 23,    # 인물당 갤러리에 저장할 이미지 수

    # 재현성을 위한 랜덤 시드
    "seed": 42,
}
# ─────────────────────────────────────────────


def sample_from_folder(cfg: dict):
    """
    폴더 형태 MS1MV3에서 50인 추출
    구조: root_dir/0000001/img001.jpg
    """
    root = Path(cfg["folder_dir"])
    output = Path(cfg["output_dir"])
    output.mkdir(parents=True, exist_ok=True)

    random.seed(cfg["seed"])
    np.random.seed(cfg["seed"])

    print("[1/4] 전체 아이덴티티 목록 수집 중...")
    all_identities = [d for d in root.iterdir() if d.is_dir()]
    print(f"      총 아이덴티티 수: {len(all_identities):,}")

    print(f"[2/4] 이미지 수 {cfg['min_images']}장 이상인 아이덴티티 필터링 중...")
    valid_identities = []
    for identity_dir in all_identities:
        images = list(identity_dir.glob("*.jpg")) + \
                 list(identity_dir.glob("*.png")) + \
                 list(identity_dir.glob("*.jpeg"))
        if len(images) >= cfg["min_images"]:
            valid_identities.append((identity_dir, images))

    print(f"      유효 아이덴티티 수: {len(valid_identities):,}")

    if len(valid_identities) < cfg["num_persons"]:
        raise ValueError(
            f"유효 아이덴티티({len(valid_identities)})가 "
            f"요청 인원({cfg['num_persons']})보다 적습니다. "
            f"min_images 값을 낮춰보세요."
        )

    print(f"[3/4] {cfg['num_persons']}인 랜덤 샘플링 중...")
    sampled = random.sample(valid_identities, cfg["num_persons"])

    print(f"[4/4] 이미지 복사 중 → {output}")
    summary = []
    for idx, (identity_dir, images) in enumerate(sampled):
        person_id = identity_dir.name
        dest_dir = output / person_id
        dest_dir.mkdir(exist_ok=True)

        # 인물당 images_per_person장 랜덤 샘플
        selected = random.sample(images, min(cfg["images_per_person"], len(images)))
        for img_path in selected:
            shutil.copy2(img_path, dest_dir / img_path.name)

        summary.append({
            "index": idx + 1,
            "identity": person_id,
            "total_available": len(images),
            "copied": len(selected),
        })
        print(f"      [{idx+1:02d}/{cfg['num_persons']}] {person_id} "
              f"({len(selected)}/{len(images)}장 복사)")

    _save_summary(summary, output, cfg)
    print(f"\n✅ 완료: {output}")
    return summary


def sample_from_rec(cfg: dict):
    """
    RecordIO(.rec) 형태 MS1MV3에서 50인 추출
    MS1MV3의 .rec 파일은 헤더에 label(identity_id)이 포함되어 있습니다.
    insightface 라이브러리의 파싱 방식을 따릅니다.
    """
    try:
        import mxnet as mx
    except ImportError:
        print("⚠️  MXNet이 설치되지 않았습니다.")
        print("   설치: pip install mxnet")
        print("   또는 폴더 형태 데이터셋을 사용하세요 (dataset_type='folder')")
        return

    rec_dir = Path(cfg["rec_dir"])
    output = Path(cfg["output_dir"])
    output.mkdir(parents=True, exist_ok=True)

    random.seed(cfg["seed"])
    np.random.seed(cfg["seed"])

    rec_path = str(rec_dir / "train.rec")
    idx_path = str(rec_dir / "train.idx")

    print("[1/5] RecordIO 파일 로딩 중...")
    record = mx.recordio.MXIndexedRecordIO(idx_path, rec_path, "r")

    print("[2/5] 아이덴티티별 이미지 인덱스 파싱 중 (시간 소요)...")
    identity_to_indices: dict[int, list[int]] = {}

    # train.idx의 키 목록으로 전체 레코드 순회
    keys = list(record.keys)
    for i, key in enumerate(keys):
        if i % 100_000 == 0:
            print(f"      파싱 중... {i:,}/{len(keys):,}")

        item = record.read_idx(key)
        header, _ = mx.recordio.unpack(item)

        # MS1MV3 헤더: label[0]이 identity id
        label = int(header.label[0]) if hasattr(header.label, "__len__") \
                else int(header.label)

        identity_to_indices.setdefault(label, []).append(key)

    print(f"      총 아이덴티티 수: {len(identity_to_indices):,}")

    print(f"[3/5] 이미지 수 {cfg['min_images']}장 이상 필터링 중...")
    valid = {
        label: indices
        for label, indices in identity_to_indices.items()
        if len(indices) >= cfg["min_images"]
    }
    print(f"      유효 아이덴티티 수: {len(valid):,}")

    print(f"[4/5] {cfg['num_persons']}인 랜덤 샘플링 중...")
    sampled_labels = random.sample(list(valid.keys()), cfg["num_persons"])

    print(f"[5/5] 이미지 디코딩 및 저장 중 → {output}")
    from PIL import Image
    import io

    summary = []
    for idx, label in enumerate(sampled_labels):
        indices = valid[label]
        selected_keys = random.sample(
            indices, min(cfg["images_per_person"], len(indices))
        )

        person_dir = output / f"identity_{label:07d}"
        person_dir.mkdir(exist_ok=True)

        for img_idx, key in enumerate(selected_keys):
            item = record.read_idx(key)
            _, img_bytes = mx.recordio.unpack(item)
            img = mx.image.imdecode(img_bytes).asnumpy()
            pil_img = Image.fromarray(img)
            pil_img.save(person_dir / f"{img_idx:04d}.jpg")

        summary.append({
            "index": idx + 1,
            "identity": f"identity_{label:07d}",
            "total_available": len(indices),
            "copied": len(selected_keys),
        })
        print(f"      [{idx+1:02d}/{cfg['num_persons']}] identity_{label:07d} "
              f"({len(selected_keys)}장 저장)")

    _save_summary(summary, output, cfg)
    print(f"\n✅ 완료: {output}")
    return summary


def _save_summary(summary: list, output: Path, cfg: dict):
    """추출 결과 요약을 텍스트 파일로 저장"""
    summary_path = output / "extraction_summary.txt"
    with open(summary_path, "w", encoding="utf-8") as f:
        f.write("=" * 50 + "\n")
        f.write("MS1MV3 50인 추출 요약\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"dataset_type   : {cfg['dataset_type']}\n")
        f.write(f"num_persons    : {cfg['num_persons']}\n")
        f.write(f"min_images     : {cfg['min_images']}\n")
        f.write(f"images_per_person: {cfg['images_per_person']}\n")
        f.write(f"random_seed    : {cfg['seed']}\n\n")
        f.write("-" * 50 + "\n")
        f.write(f"{'No':>4}  {'Identity':<20}  {'Available':>10}  {'Copied':>6}\n")
        f.write("-" * 50 + "\n")
        for row in summary:
            f.write(
                f"{row['index']:>4}  {row['identity']:<20}  "
                f"{row['total_available']:>10}  {row['copied']:>6}\n"
            )
    print(f"      요약 저장: {summary_path}")


def verify_output(output_dir: str):
    """추출 결과 검증"""
    output = Path(output_dir)
    persons = [d for d in output.iterdir() if d.is_dir()]
    print("\n── 검증 결과 ─────────────────────────")
    print(f"추출된 인물 수: {len(persons)}")
    total_images = 0
    for p in persons:
        imgs = list(p.glob("*.jpg")) + list(p.glob("*.png"))
        total_images += len(imgs)
    print(f"총 이미지 수  : {total_images}")
    print(f"인물당 평균   : {total_images / max(len(persons), 1):.1f}장")
    print("───────────────────────────────────────")


if __name__ == "__main__":
    dtype = CONFIG["dataset_type"]

    if dtype == "folder":
        summary = sample_from_folder(CONFIG)
    elif dtype == "rec":
        summary = sample_from_rec(CONFIG)
    else:
        raise ValueError(f"dataset_type은 'folder' 또는 'rec'만 가능합니다: {dtype}")

    if summary:
        verify_output(CONFIG["output_dir"])

/tmp/ipykernel_19155/3474589419.py:22: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'bool'):


[1/5] RecordIO 파일 로딩 중...
[2/5] 아이덴티티별 이미지 인덱스 파싱 중 (시간 소요)...
      파싱 중... 0/5,272,942
      파싱 중... 100,000/5,272,942
      파싱 중... 200,000/5,272,942
      파싱 중... 300,000/5,272,942
      파싱 중... 400,000/5,272,942
      파싱 중... 500,000/5,272,942
      파싱 중... 600,000/5,272,942
      파싱 중... 700,000/5,272,942
      파싱 중... 800,000/5,272,942
      파싱 중... 900,000/5,272,942
      파싱 중... 1,000,000/5,272,942
      파싱 중... 1,100,000/5,272,942
      파싱 중... 1,200,000/5,272,942
      파싱 중... 1,300,000/5,272,942
      파싱 중... 1,400,000/5,272,942
      파싱 중... 1,500,000/5,272,942
      파싱 중... 1,600,000/5,272,942
      파싱 중... 1,700,000/5,272,942
      파싱 중... 1,800,000/5,272,942
      파싱 중... 1,900,000/5,272,942
      파싱 중... 2,000,000/5,272,942
      파싱 중... 2,100,000/5,272,942
      파싱 중... 2,200,000/5,272,942
      파싱 중... 2,300,000/5,272,942
      파싱 중... 2,400,000/5,272,942
      파싱 중... 2,500,000/5,272,942
      파싱 중... 2,600,000/5,272,942
      파싱 중... 2,700,000/5,272,942
      파싱 중.

## enrollment / attack set 분할

In [3]:
import os
import shutil
import random

# ==========================================
# 1. 경로 설정 (사용자 환경에 맞게 수정)
# ==========================================
# 원본 50명 x 23장 데이터가 있는 폴더
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/ms1mv3'

# 새로 생성될 분할 폴더 경로
ENROLL_DIR = '/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/ms1mv3/MS1MV3_Enrollment'
ATTACK_DIR = '/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/ms1mv3/MS1MV3_AttackSet'

# 실험 재현성을 위해 랜덤 시드 고정 (항상 똑같이 15/8로 나뉘게 함)
random.seed(42)

# ==========================================
# 2. 분할 작업 실행
# ==========================================
# 출력 폴더가 없으면 생성
os.makedirs(ENROLL_DIR, exist_ok=True)
os.makedirs(ATTACK_DIR, exist_ok=True)

# BASE_DIR 내의 인물(ID) 폴더 리스트 가져오기
identity_folders = [f for f in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, f))]

print(f"총 {len(identity_folders)}명의 인물 폴더를 분할합니다...\n")

for identity in identity_folders:
    src_identity_path = os.path.join(BASE_DIR, identity)

    # 해당 인물 폴더 안의 이미지 목록 가져오기 (.jpg, .png 등)
    images = [img for img in os.listdir(src_identity_path) if img.lower().endswith(('.jpg', '.jpeg', '.png'))]

    if len(images) != 23:
        print(f"⚠️ 경고: '{identity}' 폴더의 이미지가 23장이 아닙니다! (현재 {len(images)}장) -> 스킵합니다.")
        continue

    # 이미지 순서를 무작위로 섞기 (편향 방지)
    random.shuffle(images)

    # 15장(Enroll) / 8장(Attack) 리스트 슬라이싱
    enroll_images = images[:15]
    attack_images = images[15:23]

    # 새 폴더 내에 인물별 하위 폴더 생성
    enroll_identity_path = os.path.join(ENROLL_DIR, identity)
    attack_identity_path = os.path.join(ATTACK_DIR, identity)
    os.makedirs(enroll_identity_path, exist_ok=True)
    os.makedirs(attack_identity_path, exist_ok=True)

    # Enrollment (15장) 복사
    for img in enroll_images:
        src_file = os.path.join(src_identity_path, img)
        dst_file = os.path.join(enroll_identity_path, img)
        shutil.copy2(src_file, dst_file) # copy2는 원본 파일의 메타데이터까지 유지합니다.

    # Attack Set (8장) 복사
    for img in attack_images:
        src_file = os.path.join(src_identity_path, img)
        dst_file = os.path.join(attack_identity_path, img)
        shutil.copy2(src_file, dst_file)

print("✅ 데이터셋 분할이 완벽하게 완료되었습니다!")
print(f"📁 갤러리 등록용(15장): {ENROLL_DIR}")
print(f"📁 공격 테스트용( 8장): {ATTACK_DIR}")

총 52명의 인물 폴더를 분할합니다...

⚠️ 경고: 'MS1MV3_Enrollment' 폴더의 이미지가 23장이 아닙니다! (현재 0장) -> 스킵합니다.
⚠️ 경고: 'MS1MV3_AttackSet' 폴더의 이미지가 23장이 아닙니다! (현재 0장) -> 스킵합니다.
✅ 데이터셋 분할이 완벽하게 완료되었습니다!
📁 갤러리 등록용(15장): /content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/ms1mv3/MS1MV3_Enrollment
📁 공격 테스트용( 8장): /content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/ms1mv3/MS1MV3_AttackSet


# 한국 연예인

## 수집

크롤링 코드는 코랩이 아닌 주피터 노트북에서 실행

## enrollment / attack set 분할

In [24]:
import os
import shutil
import random
import unicodedata  # 💡 자소 분리 해결을 위한 유니코드 라이브러리 추가

# ==========================================
# 1. 경로 설정 (캡처 화면 기준 반영)
# ==========================================
# 원본 데이터가 있는 폴더
BASE_DIR = r'/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인_전처리'

# 새로 생성될 분할 폴더 경로
ENROLL_DIR = r'/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인_전처리/한국연예인_Enrollment'
ATTACK_DIR = r'/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인_전처리/한국연예인_AttackSet'

# 실험 재현성을 위해 랜덤 시드 고정
random.seed(42)

# ==========================================
# 2. 분할 작업 실행
# ==========================================
os.makedirs(ENROLL_DIR, exist_ok=True)
os.makedirs(ATTACK_DIR, exist_ok=True)

identity_folders = [f for f in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, f))]

print(f"🔍 총 {len(identity_folders)}명의 인물 폴더를 발견했습니다.\n" + "="*50)

for identity in identity_folders:
    src_identity_path = os.path.join(BASE_DIR, identity)
    all_files = os.listdir(src_identity_path)

    valid_ext = ('.jpg', '.jpeg', '.png', '.webp')
    images = [img for img in all_files if img.lower().endswith(valid_ext)]

    if len(images) == 0:
        print(f"   ⚠️ 폴더가 비어있거나 이미지를 찾지 못했습니다. 스킵합니다.")
        continue

    # 💡 [핵심 수정] 파일명을 NFC(조합형)로 변환한 뒤 '정면', '옆모습' 글자를 찾습니다.
    frontal_images = []
    profile_images = []

    for img in images:
        # 쪼개진 한글 파일명을 정상적인 한글로 다시 조립
        normalized_name = unicodedata.normalize('NFC', img)

        if '옆모습' in normalized_name:
            profile_images.append(img) # 원본 파일명(img)을 넣어야 복사가 제대로 됩니다.
        elif '정면' in normalized_name:
            frontal_images.append(img)
        else:
            # 둘 다 글자가 없는 예외 파일들은 기본적으로 '정면'으로 간주
            frontal_images.append(img)

    print(f"👤 [{identity}] 탐색 완료: 총 {len(images)}장 (정면 {len(frontal_images)}장, 옆모습 {len(profile_images)}장)")


    # 이미지 순서를 무작위로 섞기
    random.shuffle(frontal_images)
    random.shuffle(profile_images)

    # --- [분할 핵심 로직] ---
    # 정면: 10개 Enroll / 나머지 Attack (만약 10장보다 적으면 전부 Enroll로)
    enroll_frontal = frontal_images[:10]
    attack_frontal = frontal_images[10:]

    # 옆모습: 5개 Enroll / 나머지 Attack (만약 5장보다 적으면 전부 Enroll로)
    enroll_profile = profile_images[:5]
    attack_profile = profile_images[5:]

    # Enroll과 Attack 리스트 병합
    enroll_images = enroll_frontal + enroll_profile
    attack_images = attack_frontal + attack_profile

    # 새 폴더 내에 인물별 하위 폴더 생성
    enroll_identity_path = os.path.join(ENROLL_DIR, identity)
    attack_identity_path = os.path.join(ATTACK_DIR, identity)
    os.makedirs(enroll_identity_path, exist_ok=True)
    os.makedirs(attack_identity_path, exist_ok=True)

    # Enrollment 파일 복사
    for img in enroll_images:
        src_file = os.path.join(src_identity_path, img)

        # 💡 [핵심 해결] 저장할 때는 무조건 완벽하게 합쳐진(NFC) 한글 이름으로 저장합니다.
        safe_dst_name = unicodedata.normalize('NFC', img)
        dst_file = os.path.join(enroll_identity_path, safe_dst_name)

        # 💡 copy2 대신 copy를 사용하여 클라우드 환경 메타데이터 충돌 에러 방지
        shutil.copy(src_file, dst_file)

    # Attack Set 파일 복사
    for img in attack_images:
        src_file = os.path.join(src_identity_path, img)

        safe_dst_name = unicodedata.normalize('NFC', img)
        dst_file = os.path.join(attack_identity_path, safe_dst_name)

        shutil.copy(src_file, dst_file)

    print(f"   └─ 📁 Enroll: {len(enroll_images)}장 복사 완료 | 🎯 Attack: {len(attack_images)}장 복사 완료\n")

print("="*50)
print("✅ 한국 연예인 데이터셋(정면/옆모습) 분할 및 복사가 완벽하게 완료되었습니다!")
print(f"📍 갤러리 등록용(Enrollment): {ENROLL_DIR}")
print(f"📍 공격 테스트용(AttackSet):  {ATTACK_DIR}")

🔍 총 22명의 인물 폴더를 발견했습니다.
👤 [마동석] 탐색 완료: 총 25장 (정면 16장, 옆모습 9장)
   └─ 📁 Enroll: 15장 복사 완료 | 🎯 Attack: 10장 복사 완료

👤 [류진] 탐색 완료: 총 32장 (정면 18장, 옆모습 14장)
   └─ 📁 Enroll: 15장 복사 완료 | 🎯 Attack: 17장 복사 완료

👤 [김고은] 탐색 완료: 총 29장 (정면 18장, 옆모습 11장)
   └─ 📁 Enroll: 15장 복사 완료 | 🎯 Attack: 14장 복사 완료

👤 [나재민] 탐색 완료: 총 26장 (정면 15장, 옆모습 11장)
   └─ 📁 Enroll: 15장 복사 완료 | 🎯 Attack: 11장 복사 완료

👤 [남궁민] 탐색 완료: 총 31장 (정면 18장, 옆모습 13장)
   └─ 📁 Enroll: 15장 복사 완료 | 🎯 Attack: 16장 복사 완료

👤 [비투비 이민혁] 탐색 완료: 총 29장 (정면 18장, 옆모습 11장)
   └─ 📁 Enroll: 15장 복사 완료 | 🎯 Attack: 14장 복사 완료

👤 [뷔] 탐색 완료: 총 26장 (정면 16장, 옆모습 10장)
   └─ 📁 Enroll: 15장 복사 완료 | 🎯 Attack: 11장 복사 완료

👤 [박지훈] 탐색 완료: 총 33장 (정면 15장, 옆모습 18장)
   └─ 📁 Enroll: 15장 복사 완료 | 🎯 Attack: 18장 복사 완료

👤 [설윤] 탐색 완료: 총 26장 (정면 14장, 옆모습 12장)
   └─ 📁 Enroll: 15장 복사 완료 | 🎯 Attack: 11장 복사 완료

👤 [박보검] 탐색 완료: 총 29장 (정면 12장, 옆모습 17장)
   └─ 📁 Enroll: 15장 복사 완료 | 🎯 Attack: 14장 복사 완료

👤 [송강호] 탐색 완료: 총 25장 (정면 18장, 옆모습 7장)
   └─ 📁 

# 분할 확인

In [20]:
import os

def check_dataset_split(name, enroll_dir, attack_dir, expected_enroll=None):
    print(f"==================================================")
    print(f"📊 [{name}] 데이터셋 분할 검증")
    print(f"==================================================")

    # 폴더 존재 여부 확인
    if not os.path.exists(enroll_dir) or not os.path.exists(attack_dir):
        print("❌ 지정된 폴더를 찾을 수 없습니다. 경로를 확인해주세요.")
        print(f"   Enroll: {enroll_dir}")
        print(f"   Attack: {attack_dir}\n")
        return

    # 인물 폴더 목록 가져오기 (Enroll과 Attack의 합집합)
    enroll_ids = set(os.listdir(enroll_dir))
    attack_ids = set(os.listdir(attack_dir))
    all_ids = sorted(list(enroll_ids | attack_ids))

    print(f"✅ 총 인물(Identity) 수: {len(all_ids)}명\n")

    # 콘솔 출력용 표 헤더
    print(f"{'인물 (Identity)':<18} | {'Enroll (장)':<10} | {'Attack (장)':<10} | {'총합 (장)':<8} | {'상태'}")
    print("-" * 70)

    warning_count = 0

    for identity in all_ids:
        enroll_path = os.path.join(enroll_dir, identity)
        attack_path = os.path.join(attack_dir, identity)

        # 각 폴더 내의 파일 개수 카운트
        enroll_cnt = len(os.listdir(enroll_path)) if os.path.isdir(enroll_path) else 0
        attack_cnt = len(os.listdir(attack_path)) if os.path.isdir(attack_path) else 0
        total = enroll_cnt + attack_cnt

        status = "✅ 정상"

        # 1. Enroll 장수가 15장으로 고정되어야 하는 경우
        if expected_enroll is not None:
            if enroll_cnt != expected_enroll:
                status = "⚠️ Enroll 수량 다름"
                warning_count += 1
        # 2. Enroll이 유동적이지만 아예 비어있는 경우
        else:
            if enroll_cnt == 0:
                status = "⚠️ Enroll 데이터 없음"
                warning_count += 1

        # 한글 이름의 길이를 고려해 탭(\t)과 공백을 섞어 정렬합니다.
        print(f"{identity:<18} | {enroll_cnt:<11} | {attack_cnt:<11} | {total:<10} | {status}")

    print("-" * 70)
    if warning_count == 0:
        print("🌟 완벽합니다! 모든 데이터가 조건에 맞게 성공적으로 분할되었습니다.\n\n")
    else:
        print(f"🚨 주의: 총 {warning_count}개의 인물 폴더에 확인이 필요합니다.\n\n")

# ==========================================
# 1. MS1MV3 데이터셋 검증 (Enroll 15장, Attack 8장 고정)
# ==========================================
MS1MV3_ENROLL = r'/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/ms1mv3/MS1MV3_Enrollment'
MS1MV3_ATTACK = r'/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/ms1mv3/MS1MV3_AttackSet'

check_dataset_split("MS1MV3", MS1MV3_ENROLL, MS1MV3_ATTACK, expected_enroll=15)

# ==========================================
# 2. 한국 연예인 데이터셋 검증 (정면 10+옆모습 5 등 유동적)
# ==========================================
KOR_ENROLL = r'/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인/한국연예인_Enrollment'
KOR_ATTACK = r'/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인/한국연예인_AttackSet'

check_dataset_split("한국 연예인", KOR_ENROLL, KOR_ATTACK, expected_enroll=15)

📊 [MS1MV3] 데이터셋 분할 검증
✅ 총 인물(Identity) 수: 50명

인물 (Identity)      | Enroll (장) | Attack (장) | 총합 (장)   | 상태
----------------------------------------------------------------------
identity_0000883   | 15          | 8           | 23         | ✅ 정상
identity_0003968   | 15          | 8           | 23         | ✅ 정상
identity_0004627   | 15          | 8           | 23         | ✅ 정상
identity_0005354   | 15          | 8           | 23         | ✅ 정상
identity_0005757   | 15          | 8           | 23         | ✅ 정상
identity_0008192   | 15          | 8           | 23         | ✅ 정상
identity_0008721   | 15          | 8           | 23         | ✅ 정상
identity_0013050   | 15          | 8           | 23         | ✅ 정상
identity_0014983   | 15          | 8           | 23         | ✅ 정상
identity_0015122   | 15          | 8           | 23         | ✅ 정상
identity_0016492   | 15          | 8           | 23         | ✅ 정상
identity_0017251   | 15          | 8           | 23         | ✅ 정상
identity_0017348 